Install k3s:
```bash
curl -sfL https://get.k3s.io | sh -
```

Check installation:
```bash
sudo kubectl get nodes
```

## Deploy Jitsi Meet
Jitsi Meet has 4 core components: web (frontend), prosody (XMPP server), jicofo (meeting management), and jvb (video bridge).

Run all these bash commands in your SSH terminal.

### Create a folder for Jitsi:
```bash
mkdir -p ~/k8s/jitsi
```

### Create environment for basic Jitsi configuration. Replace `A-B-C-D` with your floating IP (Make sure to use `A-B-C-D` format for nip.io links instead of `A.B.C.D`):
```bash
cat << 'EOF' > ~/k8s/jitsi/configmap.yaml
apiVersion: v1
kind: ConfigMap
metadata:
  name: jitsi-config
  namespace: jitsi
data:
  PUBLIC_URL: "https://A-B-C-D.nip.io"
  XMPP_SERVER: "prosody"
  XMPP_DOMAIN: "meet.jitsi"
  XMPP_AUTH_DOMAIN: "auth.meet.jitsi"
  XMPP_MUC_DOMAIN: "muc.meet.jitsi"
  XMPP_INTERNAL_MUC_DOMAIN: "internal-muc.meet.jitsi"
  JVB_BREWERY_MUC: "jvbbrewery"
  JVB_PORT: "10000"
  JVB_STUN_SERVERS: "meet-jit-si-turnrelay.jitsi.net:443"
  TZ: "America/New_York"
  JICOFO_AUTH_USER: "focus"
  DISABLE_HTTPS: "1"
  ENABLE_HTTP_REDIRECT: "0"
EOF
```

### Create namespace:
```bash
sudo kubectl create namespace jitsi
```

### Create k8s secret, replace `<YOUR_JICOFO-PASSWORD>` and `<YOUR-JVB-PASSWORD>` with your own passwords:
```bash
sudo kubectl create secret generic jitsi-secrets \
  --from-literal=JICOFO_AUTH_PASSWORD=<YOUR-JICOFO-PASSWORD> \  
  --from-literal=JVB_AUTH_PASSWORD=<YOUR-JVB-PASSWORD> \  
  -n jitsi
```

### Create prosody:
```bash
cat << 'EOF' > ~/k8s/jitsi/prosody.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: prosody
  namespace: jitsi
spec:
  replicas: 1
  selector:
    matchLabels:
      app: prosody
  template:
    metadata:
      labels:
        app: prosody
    spec:
      containers:
      - name: prosody
        image: jitsi/prosody:stable
        envFrom:
        - configMapRef:
            name: jitsi-config
        - secretRef:
            name: jitsi-secrets
        ports:
        - containerPort: 5222
        - containerPort: 5280
        - containerPort: 5347
        resources:
          requests:
            cpu: "50m"
            memory: "64Mi"
          limits:
            cpu: "200m"
            memory: "256Mi"
---
apiVersion: v1
kind: Service
metadata:
  name: prosody
  namespace: jitsi
spec:
  selector:
    app: prosody
  ports:
  - name: xmpp
    port: 5222
  - name: http
    port: 5280
  - name: component
    port: 5347
EOF
```

### Create jicofo:
```bash
cat << 'EOF' > ~/k8s/jitsi/jicofo.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: jicofo
  namespace: jitsi
spec:
  replicas: 1
  selector:
    matchLabels:
      app: jicofo
  template:
    metadata:
      labels:
        app: jicofo
    spec:
      containers:
      - name: jicofo
        image: jitsi/jicofo:stable
        envFrom:
        - configMapRef:
            name: jitsi-config
        - secretRef:
            name: jitsi-secrets
        ports:
        - containerPort: 8888
        resources:
          requests:
            cpu: "50m"
            memory: "200Mi"
          limits:
            cpu: "300m"
            memory: "400Mi"
---
apiVersion: v1
kind: Service
metadata:
  name: jicofo
  namespace: jitsi
spec:
  selector:
    app: jicofo
  ports:
  - name: http
    port: 8888
EOF
```

### Run prosody to get cluster ip:
```bash
sudo kubectl apply -f ~/k8s/jitsi/configmap.yaml
sudo kubectl apply -f ~/k8s/jitsi/prosody.yaml
sudo kubectl get svc -n jitsi prosody
```

### Create jvb (substitute `floating ip` and `prosody cluster ip`):
```bash
cat << 'EOF' > ~/k8s/jitsi/jvb.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: jvb
  namespace: jitsi
spec:
  replicas: 1
  selector:
    matchLabels:
      app: jvb
  template:
    metadata:
      labels:
        app: jvb
    spec:
      hostNetwork: true
      dnsPolicy: ClusterFirstWithHostNet
      containers:
      - name: jvb
        image: jitsi/jvb:stable
        envFrom:
        - configMapRef:
            name: jitsi-config
        - secretRef:
            name: jitsi-secrets
        env:
        - name: JVB_ADVERTISE_IPS
          value: "<FLOATING_IP>"
        - name: XMPP_SERVER
          value: "<PROSODY_CLUSTER_IP>"
        ports:
        - containerPort: 10000
          protocol: UDP
        - containerPort: 8080
        resources:
          requests:
            cpu: "50m"
            memory: "256Mi"
          limits:
            cpu: "500m"
            memory: "512Mi"
EOF
```

### Create web:
```bash
cat << 'EOF' > ~/k8s/jitsi/web.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: web
  namespace: jitsi
spec:
  replicas: 1
  selector:
    matchLabels:
      app: web
  template:
    metadata:
      labels:
        app: web
    spec:
      containers:
      - name: web
        image: jitsi/web:stable
        envFrom:
        - configMapRef:
            name: jitsi-config
        ports:
        - containerPort: 80
        - containerPort: 443
        resources:
          requests:
            cpu: "10m"
            memory: "32Mi"
          limits:
            cpu: "200m"
            memory: "128Mi"
---
apiVersion: v1
kind: Service
metadata:
  name: web
  namespace: jitsi
spec:
  selector:
    app: web
  ports:
  - name: http
    port: 80
    targetPort: 80
EOF
```

## Configure TLS for Jitsi Meet
Without TLS, Jitsi's WebRTC and WebSocket connection fail in modern browers, making the service unable to join meetings. We configure TLS using a self-signed certificate and K8S Ingress to enable HTTPS.

### Generate self-signed TLS certificate
Replace `A.B.C.D` with your `floating ip`:
```bash
openssl req -x509 -nodes -days 365 -newkey rsa:2048 \
  -keyout /tmp/tls.key \
  -out /tmp/tls.crt \
  -subj "/CN=A-B-C-D.nip.io"
```

### Create K8S TLS secret
```bash
sudo kubectl create secret tls jitsi-tls \
  --cert=/tmp/tls.crt \
  --key=/tmp/tls.key \
  -n jitsi
```

### Create Ingress with TLS and WebSocket routing
Replace `A-B-C-D` with your floating IP:
```bash
cat << 'EOF' > ~/k8s/jitsi/ingress.yaml
apiVersion: networking.k8s.io/v1
kind: Ingress
metadata:
  name: jitsi-ingress
  namespace: jitsi
  annotations:
    traefik.ingress.kubernetes.io/router.entrypoints: websecure
spec:
  tls:
  - hosts:
    - A-B-C-D.nip.io
    secretName: jitsi-tls
  rules:
  - host: A-B-C-D.nip.io
    http:
      paths:
      - path: /xmpp-websocket
        pathType: Prefix
        backend:
          service:
            name: prosody
            port:
              number: 5280
      - path: /
        pathType: Prefix
        backend:
          service:
            name: web
            port:
              number: 80
EOF
```

### Apply all changes and restart
```bash
sudo kubectl apply -f ~/k8s/jitsi/
sudo kubectl rollout restart deployment -n jitsi
```

### Verify deployment
```bash
sudo kubectl get all -n jitsi
```

## Access Jitsi Meet via HTTPS
Replace `A-B-C-D` with your `floating ip` and navigate to the link in your brower, accept the self-signed certificate warning  to acccess Jitsi Meet:  

https://A-B-C-D.nip.io